In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

import xgboost as xgb
import optuna

In [2]:
train_df = pd.read_csv("training_bob(2003-2023).csv")
test_df = pd.read_csv("test_bob(2024-25).csv")

print(train_df.shape)
print(test_df.shape)

(3375208, 14)
(176241, 14)


In [3]:
FEATURES = [
    "CHLOR_A",
    "day_sin",
    "day_cos",
    "month_sin",
    "month_cos",
    "LAT_scaled",
    "LON_scaled"
]

TARGET = "PHYTOBLOOM"

In [4]:
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

In [8]:

import joblib

final_model =  joblib.load('lgb_model_BOB.pkl')

final_model.fit(
    X_train,
    y_train
)

[LightGBM] [Warning] Unknown parameter: weight_mode
[LightGBM] [Warning] Unknown parameter: weight_mode
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027239 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 869
[LightGBM] [Info] Number of data points in the train set: 3375208, number of used features: 7
[LightGBM] [Info] Start training from score -0.039413
[LightGBM] [Info] Start training from score -4.385591
[LightGBM] [Info] Start training from score -3.642329
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Light

,num_leaves,65
,max_depth,5
,learning_rate,0.025754508160065227
,n_estimators,320
,objective,'multiclass'
,min_split_gain,0.47128768439445706
,min_child_samples,46
,subsample,0.6815372336035705
,colsample_bytree,0.937585105687456
,reg_alpha,0.24712644463561262
,reg_lambda,0.047393210901825036


In [9]:
y_pred1 = final_model.predict(
    X_train
)
y_pred = final_model.predict(
    X_test
)

[LightGBM] [Warning] Unknown parameter: weight_mode
[LightGBM] [Warning] Unknown parameter: weight_mode


In [10]:
print("TRAIN RESULTS")
print(
    classification_report(
        y_train,
        y_pred1
    )
)

macro_f1 = f1_score(
    y_train,
    y_pred1,
    average="macro"
)

print(
    "Train Macro F1:",
    macro_f1
)
print()
print("TEST RESULTS")
print(
    classification_report(
        y_test,
        y_pred
    )
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print(
    "Test Macro F1:",
    macro_f1
)

TRAIN RESULTS
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99   3244767
         1.0       0.71      0.65      0.68     42040
         2.0       0.73      0.68      0.70     88401

    accuracy                           0.98   3375208
   macro avg       0.81      0.77      0.79   3375208
weighted avg       0.98      0.98      0.98   3375208

Train Macro F1: 0.7907327666055178

TEST RESULTS
              precision    recall  f1-score   support

         0.0       0.98      0.98      0.98    162353
         1.0       0.62      0.66      0.64      5693
         2.0       0.82      0.68      0.74      8195

    accuracy                           0.96    176241
   macro avg       0.81      0.77      0.79    176241
weighted avg       0.96      0.96      0.96    176241

Test Macro F1: 0.7876709935352686


In [5]:
def objective_multi(trial):
    # params = {
    #     "objective": "multi:softprob",
    #     "num_class": 3,

    #     "n_estimators": trial.suggest_int("n_estimators", 20, 90, step=5),
    #     "max_depth": trial.suggest_int("max_depth", 4, 9),
    #     "learning_rate": trial.suggest_float("learning_rate", 0.08, 0.25, log=True),
    #     "subsample": trial.suggest_float("subsample", 0.65, 0.85),
    #     "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.88),
    #     "min_child_weight": trial.suggest_int("min_child_weight", 15, 30),
    #     "gamma": trial.suggest_float("gamma", 0.02, 1.0, log=True),
    #     "lambda": trial.suggest_float("lambda", 0.3, 5.0, log=True),
    #     "alpha": trial.suggest_float("alpha", 1.6, 2.8),
    #     "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),

    #     "tree_method": "hist",
    #     "random_state": 42,
    #     "n_jobs": -1,
    # }
    params = {
        "objective": "multi:softprob",
        "num_class": 3,

        "n_estimators": trial.suggest_int("n_estimators", 30, 200, step=10),
        "max_depth": trial.suggest_int("max_depth", 3, 14),  # narrowed hard around 4
        "learning_rate": trial.suggest_float("learning_rate", 0.10, 0.24, log=True),
        "subsample": trial.suggest_float("subsample", 0.70, 0.85),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.85),
        "min_child_weight": trial.suggest_int("min_child_weight", 18, 30),
        "gamma": trial.suggest_float("gamma", 0.05, 0.5, log=True),
        "lambda": trial.suggest_float("lambda", 0.4, 2.5, log=True),
        "alpha": trial.suggest_float("alpha", 1.7, 2.8),
        "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),

        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1,
    }


    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gaps, val_scores = [], []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr, verbose=False)

        train_f1 = f1_score(y_tr, model.predict(X_tr), average="macro")
        val_f1 = f1_score(y_val, model.predict(X_val), average="macro")

        val_scores.append(val_f1)
        gaps.append(train_f1 - val_f1)

    mean_val = np.mean(val_scores)
    mean_gap = np.mean(gaps)

    # Return a tuple: optuna will treat this as two objectives
    return mean_val, mean_gap

In [6]:
study = optuna.create_study(directions=["maximize", "minimize"]) 
study.optimize(objective_multi, n_trials=40, show_progress_bar=True)

[I 2026-07-12 21:44:37,708] A new study created in memory with name: no-name-f877c2fb-97d4-4819-9c0f-693de4aa66ce


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-07-12 21:47:33,258] Trial 0 finished with values: [0.8581922835963999, 0.017513507120277837] and parameters: {'n_estimators': 60, 'max_depth': 13, 'learning_rate': 0.2101037505552234, 'subsample': 0.714965324771026, 'colsample_bytree': 0.7263591568220562, 'min_child_weight': 23, 'gamma': 0.35864938303199506, 'lambda': 1.7826314745465124, 'alpha': 1.764000846825751, 'grow_policy': 'depthwise'}.
[I 2026-07-12 21:55:15,321] Trial 1 finished with values: [0.8401493533183325, 0.01028272174805167] and parameters: {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.15824281502129184, 'subsample': 0.8266483208597931, 'colsample_bytree': 0.8086674306336382, 'min_child_weight': 27, 'gamma': 0.1235343717892182, 'lambda': 1.4222620125149268, 'alpha': 2.6458833222408, 'grow_policy': 'depthwise'}.
[I 2026-07-12 22:00:17,160] Trial 2 finished with values: [0.7901342908282145, 0.0027905426635567387] and parameters: {'n_estimators': 140, 'max_depth': 4, 'learning_rate': 0.15194210358020652

In [7]:
best_trials = study.best_trials
i=0

for t in best_trials:
    print(f"{i} val_f1={t.values[0]:.4f}, gap={t.values[1]:.4f}, params={t.params}")
    i+=1

0 val_f1=0.8582, gap=0.0175, params={'n_estimators': 60, 'max_depth': 13, 'learning_rate': 0.2101037505552234, 'subsample': 0.714965324771026, 'colsample_bytree': 0.7263591568220562, 'min_child_weight': 23, 'gamma': 0.35864938303199506, 'lambda': 1.7826314745465124, 'alpha': 1.764000846825751, 'grow_policy': 'depthwise'}
1 val_f1=0.8401, gap=0.0103, params={'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.15824281502129184, 'subsample': 0.8266483208597931, 'colsample_bytree': 0.8086674306336382, 'min_child_weight': 27, 'gamma': 0.1235343717892182, 'lambda': 1.4222620125149268, 'alpha': 2.6458833222408, 'grow_policy': 'depthwise'}
2 val_f1=0.7901, gap=0.0028, params={'n_estimators': 140, 'max_depth': 4, 'learning_rate': 0.15194210358020652, 'subsample': 0.848165338891208, 'colsample_bytree': 0.8324137910993592, 'min_child_weight': 23, 'gamma': 0.06990025130993241, 'lambda': 1.6490322984722285, 'alpha': 2.706720980659049, 'grow_policy': 'lossguide'}
3 val_f1=0.8698, gap=0.0246, pa

In [8]:
i=0
for t in best_trials:
    params=t.params
    lgb_model = xgb.XGBClassifier(
    objective='multiclass',
    num_class=3,
    **params,
    random_state=42
    )

    lgb_model.fit(X_train, y_train)
    y_pred1 = lgb_model.predict(
        X_train
    )

    y_pred = lgb_model.predict(
        X_test
    )
    macro_f1_train = f1_score(
        y_train,
        y_pred1,
        average="macro"
    )

    macro_f1_test = f1_score(
        y_test,
        y_pred,
        average="macro"
    )
    gap=macro_f1_train-macro_f1_test
    print(f"{i} Gap: {gap} Train Macro F1 : {macro_f1_train}  Test Macro F1 : {macro_f1_test}")
    i+=1


0 Gap: 0.11301799082359887 Train Macro F1 : 0.8767771397750769  Test Macro F1 : 0.763759148951478
1 Gap: 0.07607617287506507 Train Macro F1 : 0.8504491073759777  Test Macro F1 : 0.7743729345009126
2 Gap: 0.003985611884747997 Train Macro F1 : 0.7937912277273257  Test Macro F1 : 0.7898056158425777
3 Gap: 0.13237637835078964 Train Macro F1 : 0.8959897891651692  Test Macro F1 : 0.7636134108143796
4 Gap: 0.09058469837194705 Train Macro F1 : 0.8635955527453154  Test Macro F1 : 0.7730108543733684
5 Gap: 0.13093839000368623 Train Macro F1 : 0.8935770819346534  Test Macro F1 : 0.7626386919309671
6 Gap: 0.11476534372840619 Train Macro F1 : 0.8827078393588682  Test Macro F1 : 0.767942495630462
7 Gap: -0.022565160319755306 Train Macro F1 : 0.7586670359725664  Test Macro F1 : 0.7812321962923217
8 Gap: 0.029906920934108294 Train Macro F1 : 0.7997808710735316  Test Macro F1 : 0.7698739501394233
9 Gap: 0.15307903072279982 Train Macro F1 : 0.9078505246027649  Test Macro F1 : 0.7547714938799651
10 Gap: 

In [12]:
params=best_trials[2].params
params

{'n_estimators': 140,
 'max_depth': 4,
 'learning_rate': 0.15194210358020652,
 'subsample': 0.848165338891208,
 'colsample_bytree': 0.8324137910993592,
 'min_child_weight': 23,
 'gamma': 0.06990025130993241,
 'lambda': 1.6490322984722285,
 'alpha': 2.706720980659049,
 'grow_policy': 'lossguide'}

In [5]:
params={'n_estimators': 140,
 'max_depth': 4,
 'learning_rate': 0.15194210358020652,
 'subsample': 0.848165338891208,
 'colsample_bytree': 0.8324137910993592,
 'min_child_weight': 23,
 'gamma': 0.06990025130993241,
 'lambda': 1.6490322984722285,
 'alpha': 2.706720980659049,
 'grow_policy': 'lossguide'}

In [8]:
lgb_model = xgb.XGBClassifier(
    objective='multiclass',
    num_class=3,
    **params,
    random_state=42
    )

lgb_model.fit(X_train, y_train)
y_pred1 = lgb_model.predict(
    X_train
)

y_pred = lgb_model.predict(
    X_test
)
macro_f1_train = f1_score(
    y_train,
    y_pred1,
    average="macro"
)

macro_f1_test = f1_score(
    y_test,
    y_pred,
    average="macro"
)
gap=macro_f1_train-macro_f1_test
print(f"Gap: {gap} Train Macro F1 : {macro_f1_train}  Test Macro F1 : {macro_f1_test}")

Gap: 0.003985611884747997 Train Macro F1 : 0.7937912277273257  Test Macro F1 : 0.7898056158425777


In [9]:
import joblib

# Save the model
joblib.dump(lgb_model, 'xgb_model_BOB.pkl')

['xgb_model_BOB.pkl']